In [1]:
import illustris_python as il
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append('..')

from astropy.cosmology import FlatLambdaCDM
import params_icl as P
import simulation_physical_improves as spi


%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 6), 
              'figure.dpi': 150,
              'font.size': 14,
              'font.family': 'serif',
              'axes.labelsize': 14})

cosmo = FlatLambdaCDM(H0=P.H0, Om0=P.Om0)
h = P.H0/100

## catalogo de halos masivos

In [2]:
fields_halos = P.HALOS_FIELDS
fields_subhalos = P.SUBHALOS_FIELDS 

In [7]:
subhalos = il.groupcat.loadSubhalos(P.basePath, P.SNAP, fields_subhalos) 
subhalos['SubfindID'] = np.arange(subhalos['SubhaloMassType'][:, 4].shape[0])

stellar_masses_subhalos = np.log10(subhalos['SubhaloMassType'][:, 4]/h) + 10
flag_subhalos = subhalos['SubhaloFlag'] == 1

valid_subhalos = stellar_masses_subhalos >= 9.5

#valid_subhalos = subhalos['SubhaloMassType'][stellar_subhalos > 9.5]

/tmp/ipykernel_12101/446849245.py:4: RuntimeWarning: divide by zero encountered in log10
  stellar_masses_subhalos = np.log10(subhalos['SubhaloMassType'][:, 4]/h) + 10


In [ ]:
halos = il.groupcat.loadHalos(P.basePath, P.SNAP, fields_halos)
halos['HaloID'] = np.arange(halos['Group_M_Crit200'].shape[0])

masses_halos = np.log10(halos['Group_M_Crit200']/h) + 10
valid_halos = masses_halos >= 13



/tmp/ipykernel_12101/3096156081.py:3: RuntimeWarning: divide by zero encountered in log10
  masses_halos = np.log10(halos['Group_M_Crit200']/h) + 10


In [11]:
subhalos_tree = il.groupcat.loadSubhalos(P.basePath, P.SNAP, fields_subhalos)
halos_tree = il.groupcat.loadHalos(P.basePath, P.SNAP, fields_halos)

In [ ]:
# catalog cookin
F05R200, F05R200_deltamags, F05_distance, F05_mass, F05_radius, F05_sats = [], [], [], [], [], []
NF05R200, NF05R200_deltamags, NF05_distance, NF05_mass, NF05_radius, NF05_sats = [], [], [], [], [], []

F1R200, F1R200_deltamags, F1_distance, F1_mass, F1_radius, F1_sats = [], [], [], [], [], []
NF1R200, NF1R200_deltamags, NF1_distance, NF1_mass, NF1_radius, NF1_sats = [], [], [], [], [], []

iso_fossil_05, iso_F05_mass, iso_F05_radius, iso_F05_sats = [], [], [], []
iso_fossil_1, iso_F1_mass, iso_F1_radius, iso_F1_sats = [], [], [], []

In [16]:
# conditions to separate the sample 

stellar_mass = np.log10(subhalos['SubhaloMassType'][:, 4]/h) + 10
pos = (subhalos_tree['SubhaloPos'][:]/h)
R200 = halos_tree['Group_R_Crit200'][:]/h
M200 = halos_tree['Group_M_Crit200'][:]/h
mag_r = subhalos['SubhaloStellarPhotometrics'][:, 5]
cens =  halos_tree['GroupFirstSub'][:]
subhalos_GrNr = subhalos_tree['SubhaloGrNr'][:]
ID_subhalos = subhalos['SubfindID']
#satellites with mass threshold: 
#ID_mass_treshold = subhalos['SubfindID'][valid_subhalos])]

/tmp/ipykernel_12101/3342055873.py:3: RuntimeWarning: divide by zero encountered in log10
  stellar_mass = np.log10(subhalos['SubhaloMassType'][:, 4]/h) + 10


In [ ]:
for i in range(len(valid_halos)):
    group = halos['haloID'][valid_halos][i]
    central = cens[group]
    
    tree_first = il.sublink.loadTree(P.basePath, P.SNAP, id=central, fields=['SubfindID', 'SnapNum'], onlyMPB=True)
    
    halo_check = subhalos_GrNr[tree_first['SubfindID'][0]]
    
    if central != cens[halo_check]:
        print('halo mismatch')
        continue
    print(halo_check)
    
    # stellar mass cut
    gals = np.where((subhalos_GrNr == group) & (stellar_mass > 9.5))[0]
    
    central_pos = pos[central]
    r200_halo = R200[halo_check]
    m200_group = M200[group]
    #r200_group = R200[group]
    
    xc = spi.Distance_1D(pos[gals, 0], central_pos, (50000/h))
    yc = spi.Distance_1D(pos[gals, 1], central_pos, (50000/h))
    zc = spi.Distance_1D(pos[gals, 2], central_pos, (50000/h))
    normpos = np.sqrt(xc**2 + yc**2 + zc**2) / r200_halo
    
    print('-halo: ', group, '\n -central: ', central, '\n -stellar mass: ', stellar_mass[central], '\n -r200: ', r200_halo, '\n -m200: ', m200_group)
    
    mags_all = subhalos_tree['SubhaloStellarPhotometrics'][gals, 5]
    central_galaxy = np.where(normpos == 0)[0]
    central_mag = mags_all[central_galaxy]
    
    for r_frac, F_lists, NF_lists, iso_list, iso_mass, iso_radius in zip(
        [0.5, 1.0],
        [(F05r200, F05r200deltamags, F05distance, F05mass, F05radius, F05sats),
         (Fr200,   Fr200deltamags,   F1distance,  F1mass,  F1radius,  F1sats)]
        [(NF05r200, NF05r200deltamags, NF05distance, NF05mass, NF05radius, NF05sats),
         (NFr200,   NFr200deltamags,   NF1distance,  NF1mass,  NF1radius,  NF1sats)]
        [iso_fossil_05, iso_fossil_1, iso_fossil_3],
        [iso_F05_mass,  iso_F1_mass,  iso_F3_mass]):
            if not spi.classify_fossil(group, mags_all, central_mag, normpos, gals,
                            m200_group, r200_group, F_lists, NF_lists,
                            r_frac=r_frac):
                print(f'El grupo {group} solo tiene una galaxia central dentro de {r_frac} R200')
                iso_list.append(group)
                iso_mass.append(m200_group)
                iso_radius.append(r200_group)

In [ ]:
output_file = 'fossil_cat_TNG50.hdf5'

with h5py.File(output_file, 'w') as f:
    
    f.attrs['description'] = 'Catalog of fossil groups in TNG50'
    f.attrs['snapshot'] = P.SNAP
    f.attrs['gap_threshold'] = P.GAP_THRESHOLD
    f.attrs['mass_cut_log10'] = 9.5
    f.attrs['radii_R200'] = [0.5, 1.0]
    
    for r_label, F_data, NF_data, iso_data in zip(
            ['0.5R200', '1.0R200'],
            [(F05r200, F05r200deltamags, F05distance, F05mass, F05radius, F05sats),
            (Fr200,   Fr200deltamags,   F1distance,  F1mass,  F1radius,  F1sats)],
            [(NF05r200, NF05r200deltamags, NF05distance, NF05mass, NF05radius, NF05sats),
            (NFr200,   NFr200deltamags,   NF1distance,  NF1mass,  NF1radius,  NF1sats)],
            [iso_fossil_05, iso_fossil_1]):
        r_group = f.create_group(r_label)
        spi.save_category(r_group.create_group('Fossil'), *F_data)
        spi.save_category(r_group.create_group('NonFossil'), *NF_data)
        spi.save_isolated(r_group.create_group('Isolated'), *iso_data)
        
print(f'Catalog saved to {output_file}')
